# Setup

In [1]:
import os
import hail as hl
import pandas as pd
hl.init(default_reference='GRCh38', idempotent=True)

Loading BokehJS ...

/opt/conda/envs/jupyter/lib/python3.12/site-packages/hail/context.py:350: UserWarning:

Using hl.init with a default_reference argument is deprecated. To set a default reference genome after initializing hail, call `hl.default_reference` with an argument to set the default reference genome.



2026-08-20 16:40:39.272 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/opt/conda/envs/jupyter/lib/python3.12/site-packages/hail/backend/py4j_backend.py:75: UserWarning:

Hail was built and tested with Java 11. You are using Java 17.0.19 which is not supported. This may lead to errors. Consider installing Java 11 and setting the JAVA_HOME environment variable.

2026-08-20 16:40:43.961 SparkBackend$: WARN: This Hail JAR was compiled for Spark 3.5.3, running with Spark 3.5.8.
  Compatibility is not guaranteed.
2026-08-20 16:40:44.072 SparkConf: WARN: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).
Running on Apache Spark version 3.5.8
SparkUI available at http://f9a587222c54:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.137-733ac4ccd943
LOGGING: writing

In [2]:
import requests

url = "https://api.github.com/repos/broadinstitute/eMERGE-implemented-PRS-models-Lennon-et-al/contents/prs_files"
params = {"ref": "main"}

r = requests.get(url, params=params)
r.raise_for_status()

files = r.json()

for item in files:
    if item["type"] == "file":
        print(item["name"])

afib_sites_used_in_score.ids
ast_sites_used_in_score.ids
asthma_PRS_weights.txt
atrial_fibrilation_PRS_weights.txt
brca_sites_used_in_score.ids
breast_cancer_PRS_weights.txt
chd_sites_used_in_score.ids
ckd_PRS_weights.txt
ckd_apol1_risk_alleles.txt
ckd_sites_used_in_score.ids
coronary_heart_disease_PRS_weights.txt
hcl_sites_used_in_score.ids
hypercholesterolemia_PRS_weights.txt
prca_sites_used_in_score.ids
prostate_cancer_PRS_weights.txt
t1d_PRS_weights.txt
t1d_hla_interactions.tsv
t1d_hla_interactions_self_exclusive_sites.tsv
t1d_sites_used_in_score.ids
t2d_PRS_weights.txt
t2d_sites_used_in_score.ids


## Pull in weights files from github

In [3]:
import requests
from pathlib import Path

api_url = "https://api.github.com/repos/broadinstitute/eMERGE-implemented-PRS-models-Lennon-et-al/contents/prs_files"
params = {"ref": "main"}

dest_dir = Path("/home/jupyter/workspace/rw-migration-aou-rw-67cadd48/weights/hg19")
dest_dir.mkdir(parents=True, exist_ok=True)

r = requests.get(api_url, params=params)
r.raise_for_status()

files = r.json()

weights_files = [
    item for item in files
    if item["type"] == "file" and "weights" in item["name"].lower()
]

for item in weights_files:
    out_path = dest_dir / item["name"]

    resp = requests.get(item["download_url"])
    resp.raise_for_status()

    out_path.write_bytes(resp.content)
    print(f"Downloaded {item['name']} -> {out_path}")

Downloaded asthma_PRS_weights.txt -> /home/jupyter/workspace/rw-migration-aou-rw-67cadd48/weights/hg19/asthma_PRS_weights.txt
Downloaded atrial_fibrilation_PRS_weights.txt -> /home/jupyter/workspace/rw-migration-aou-rw-67cadd48/weights/hg19/atrial_fibrilation_PRS_weights.txt
Downloaded breast_cancer_PRS_weights.txt -> /home/jupyter/workspace/rw-migration-aou-rw-67cadd48/weights/hg19/breast_cancer_PRS_weights.txt
Downloaded ckd_PRS_weights.txt -> /home/jupyter/workspace/rw-migration-aou-rw-67cadd48/weights/hg19/ckd_PRS_weights.txt
Downloaded coronary_heart_disease_PRS_weights.txt -> /home/jupyter/workspace/rw-migration-aou-rw-67cadd48/weights/hg19/coronary_heart_disease_PRS_weights.txt
Downloaded hypercholesterolemia_PRS_weights.txt -> /home/jupyter/workspace/rw-migration-aou-rw-67cadd48/weights/hg19/hypercholesterolemia_PRS_weights.txt
Downloaded prostate_cancer_PRS_weights.txt -> /home/jupyter/workspace/rw-migration-aou-rw-67cadd48/weights/hg19/prostate_cancer_PRS_weights.txt
Download

# Prepare for liftover

In [4]:
import urllib.request

chain_path = Path("/home/jupyter/workspace/rw-migration-aou-rw-67cadd48/weights/grch37_to_grch38.over.chain.gz")


urllib.request.urlretrieve(
        "https://storage.googleapis.com/hail-common/references/grch37_to_grch38.over.chain.gz",
        chain_path
    )

rg37 = hl.get_reference('GRCh37')
rg38 = hl.get_reference('GRCh38')  

rg37.add_liftover(str(chain_path), rg38)

## Harmonize file colnames

In [8]:
from pathlib import Path

hg19_dir = Path("/home/jupyter/workspace/rw-migration-aou-rw-67cadd48/weights/hg19")

dfs = {}

for file in weights_files:
    filename = file["name"]
    path = hg19_dir / filename

    df = pd.read_csv(path, sep=None, engine="python")
    dfs[filename] = df

    print(filename)
    display(df.head())

asthma_PRS_weights.txt


,CHR:BP:A1:A2,A1,BETA,CHR,BP,A1.1,A2
0,1:779322:A:G,G,0.000017,1,779322,G,A
1,1:1005806:C:T,T,-0.000277,1,1005806,T,C
2,1:1017197:C:T,C,-0.000066,1,1017197,C,T
3,1:1017587:C:T,T,-0.000139,1,1017587,T,C
4,1:1018704:A:G,A,-0.000159,1,1018704,A,G


atrial_fibrilation_PRS_weights.txt


,chr_name:chr_position:effect_allele:reference_allele,effect_allele,effect_weight,chr_name,chr_position,effect_allele.1,reference_allele
0,1:10790797:A:C,C,0.0422,1,10790797,C,A
1,1:22282619:C:T,C,0.0621,1,22282619,C,T
2,1:41544279:G:T,G,0.0439,1,41544279,G,T
3,1:49309764:A:G,A,0.1456,1,49309764,A,G
4,1:51535039:A:G,A,0.1605,1,51535039,A,G


breast_cancer_PRS_weights.txt


...

,chr_name:chr_position:effect_allele:reference_allele,effect_allele,effect_weight,chr_name,chr_position,effect_allele.1,reference_allele
0,1:5743196:C:T,T,0.102298,1,5743196,T,C
1,1:10564675:A:G,A,0.042411,1,10564675,A,G
2,1:16376831:C:T,C,0.055507,1,16376831,C,T
3,1:46251655:C:T,T,0.072822,1,46251655,T,C
4,1:88210715:A:AT,AT,0.048256,1,88210715,AT,A


t1d_PRS_weights.txt


,id,allele,weight
0,6:32411712:A:G,A,-2.21
1,6:32637121:A:G,A,-0.63
2,6:32602751:A:C,A,-1.43
3,6:32603212:C:T,C,-0.03
4,6:32138545:A:C,A,-1.39


t2d_PRS_weights.txt


,CHR:BP:A1:A2,A1,BETA,CHR,BP,A1.1,A2
0,1:752721:A:G,A,-0.000114,1,752721,A,G
1,1:754182:A:G,A,-0.000142,1,754182,A,G
2,1:760912:C:T,C,0.000005,1,760912,C,T
3,1:768448:A:G,A,0.000011,1,768448,A,G
4,1:779322:A:G,G,-0.000060,1,779322,G,A


In [16]:
new_cols = ["id", "effect_allele", "weight", "chr", "pos", "a1", "a2"]

# For T1D we need to make some columns by splitting id
df = dfs["t1d_PRS_weights.txt"].copy()

parts = df["id"].str.split(":", expand=True)

df["chr"] = parts[0].astype(int)
df["pos"] = parts[1].astype(int)
df["a1"] = parts[2]
df["a2"] = parts[3]

df.head()

df = df[["id", "allele", "weight", "chr", "pos", "a1", "a2"]]

dfs["t1d_PRS_weights.txt"] = df

dfs["t1d_PRS_weights.txt"].head()

for filename, df in dfs.items():
        df.columns = new_cols
        
for filename, df in dfs.items():
    print(filename)
    print(df.columns.tolist())

asthma_PRS_weights.txt
['id', 'effect_allele', 'weight', 'chr', 'pos', 'a1', 'a2']
atrial_fibrilation_PRS_weights.txt
['id', 'effect_allele', 'weight', 'chr', 'pos', 'a1', 'a2']
breast_cancer_PRS_weights.txt
['id', 'effect_allele', 'weight', 'chr', 'pos', 'a1', 'a2']
ckd_PRS_weights.txt
['id', 'effect_allele', 'weight', 'chr', 'pos', 'a1', 'a2']
coronary_heart_disease_PRS_weights.txt
['id', 'effect_allele', 'weight', 'chr', 'pos', 'a1', 'a2']
hypercholesterolemia_PRS_weights.txt
['id', 'effect_allele', 'weight', 'chr', 'pos', 'a1', 'a2']
prostate_cancer_PRS_weights.txt
['id', 'effect_allele', 'weight', 'chr', 'pos', 'a1', 'a2']
t1d_PRS_weights.txt
['id', 'effect_allele', 'weight', 'chr', 'pos', 'a1', 'a2']
t2d_PRS_weights.txt
['id', 'effect_allele', 'weight', 'chr', 'pos', 'a1', 'a2']


# Liftover locus

In [20]:
from pathlib import Path
import hail as hl

tmp_dir = Path("/home/jupyter/workspace/rw-migration-aou-rw-67cadd48/weights/tmp_hg19_harmonized")
hg38_dir = Path("/home/jupyter/workspace/rw-migration-aou-rw-67cadd48/weights/hg38")

tmp_dir.mkdir(parents=True, exist_ok=True)
hg38_dir.mkdir(parents=True, exist_ok=True)

for filename, df in dfs.items():
    df = df.copy()

    df["chr"] = df["chr"].astype(str)
    df["pos"] = df["pos"].astype(int)
    df["weight"] = df["weight"].astype(float)

    tmp_path = tmp_dir / filename
    out_path = hg38_dir / filename

    df.to_csv(tmp_path, sep="\t", index=False)

    weights = hl.import_table(
        str(tmp_path),
        delimiter="\t",
        types={
            "id": hl.tstr,
            "effect_allele": hl.tstr,
            "weight": hl.tfloat64,
            "chr": hl.tstr,
            "pos": hl.tint32,
            "a1": hl.tstr,
            "a2": hl.tstr,
        },
    )

    weights = weights.annotate(
        locus_hg19=hl.locus(weights.chr, weights.pos, reference_genome="GRCh37")
    )

    weights = weights.annotate(
        locus_hg38=hl.liftover(weights.locus_hg19, "GRCh38")
    )

    n_orig = weights.count()
    n_lifted = weights.aggregate(
        hl.agg.count_where(hl.is_defined(weights.locus_hg38))
    )

    weights = weights.filter(hl.is_defined(weights.locus_hg38))

    weights = weights.annotate(
        chr=weights.locus_hg38.contig,
        pos=weights.locus_hg38.position,
        id=hl.variant_str(weights.locus_hg38, hl.sorted([weights.a1, weights.a2])),
    )

    weights.drop("locus_hg19", "locus_hg38").export(
        str(out_path),
        delimiter=","
    )

    print(f"{filename}: {n_lifted}/{n_orig} ({100*n_lifted/n_orig:.2f}%) sites successfully lifted over")
    print(f"lifted over weights file written to {out_path}")

asthma_PRS_weights.txt: 985494/985837 (99.97%) sites successfully lifted over
lifted over weights file written to /home/jupyter/workspace/rw-migration-aou-rw-67cadd48/weights/hg38/asthma_PRS_weights.txt
atrial_fibrilation_PRS_weights.txt: 163/163 (100.00%) sites successfully lifted over
lifted over weights file written to /home/jupyter/workspace/rw-migration-aou-rw-67cadd48/weights/hg38/atrial_fibrilation_PRS_weights.txt


breast_cancer_PRS_weights.txt: 309/309 (100.00%) sites successfully lifted over
lifted over weights file written to /home/jupyter/workspace/rw-migration-aou-rw-67cadd48/weights/hg38/breast_cancer_PRS_weights.txt


ckd_PRS_weights.txt: 471119/471316 (99.96%) sites successfully lifted over
lifted over weights file written to /home/jupyter/workspace/rw-migration-aou-rw-67cadd48/weights/hg38/ckd_PRS_weights.txt


coronary_heart_disease_PRS_weights.txt: 460482/460482 (100.00%) sites successfully lifted over
lifted over weights file written to /home/jupyter/workspace/rw-migration-aou-rw-67cadd48/weights/hg38/coronary_heart_disease_PRS_weights.txt


hypercholesterolemia_PRS_weights.txt: 9008/9009 (99.99%) sites successfully lifted over
lifted over weights file written to /home/jupyter/workspace/rw-migration-aou-rw-67cadd48/weights/hg38/hypercholesterolemia_PRS_weights.txt
prostate_cancer_PRS_weights.txt: 269/269 (100.00%) sites successfully lifted over
lifted over weights file written to /home/jupyter/workspace/rw-migration-aou-rw-67cadd48/weights/hg38/prostate_cancer_PRS_weights.txt


t1d_PRS_weights.txt: 67/67 (100.00%) sites successfully lifted over
lifted over weights file written to /home/jupyter/workspace/rw-migration-aou-rw-67cadd48/weights/hg38/t1d_PRS_weights.txt


t2d_PRS_weights.txt: 1259297/1259754 (99.96%) sites successfully lifted over
lifted over weights file written to /home/jupyter/workspace/rw-migration-aou-rw-67cadd48/weights/hg38/t2d_PRS_weights.txt
